# Project Description: 
this project is of forecasting the furture remttence that neapl is going to get in upcomming future. 
so what is the problem and what is my solution and what can be the motive of my forecasting , 
- what model am i going to use and why 

In [3]:
import pandas as pd 

The source data from Nepal Rastra Bank isn't stored as one row per month like a normal dataset. It's stored as a report table — each row is a financial category (like 'Workers' Remittances'), and each column is a specific month of a specific fiscal year. I had to reshape this into a proper time series before I could do anything else.

In [13]:
df5_raw = pd.read_excel("Balance-of-Payments-BPM5.xlsx", header=None)
df6_raw = pd.read_excel("Balance-of-Payments-BPM6.xlsx", header=None)

In [14]:
# BPM5: single column per month, target row = Excel row 32 -> pandas index 31
bpm5_target_row = df5_raw.iloc[31]

# BPM6: Credit/Debit/Net triplet per month, target row = Excel row 56 -> pandas index 55
bpm6_target_row = df6_raw.iloc[55]

print(df5_raw.iloc[31, 0:2])   # confirm the row label reads "Workers' remittances"
print(df6_raw.iloc[55, 0:2])   # confirm the row label reads "O/W Workers' remittances"

0    NaN
1    NaN
Name: 31, dtype: object
0                   1.C.2.1.1
1    O/W Workers' remittances
Name: 55, dtype: object


In [15]:
def extract_bpm5(df_raw):
    rows = []
    fy_label = None
    for c in range(5, df_raw.shape[1]):          # data starts col index 5 ('Aug' of first FY)
        y = df_raw.iloc[2, c]
        if pd.notna(y):
            fy_label = y
        month = df_raw.iloc[3, c]
        if pd.isna(month):
            continue
        val = df_raw.iloc[31, c]
        if pd.isna(val):
            continue
        rows.append({"fiscal_year": str(fy_label).strip(),
                     "month": str(month).strip(),
                     "cumulative_npr_million": float(val)})
    return pd.DataFrame(rows)

bpm5 = extract_bpm5(df5_raw)
bpm5.head()

,fiscal_year,month,cumulative_npr_million
0,2000/01,Aug,2445.9
1,2000/01,Sep,5295.8
2,2000/01,Oct,11452.4
3,2000/01,Nov,15370.7
4,2000/01,Dec,19230.4


In [16]:
def extract_bpm6(df_raw):
    rows = []
    fy_label = None
    c = 2   # first Credit column
    while c < df_raw.shape[1]:
        y = df_raw.iloc[2, c]
        if pd.notna(y):
            fy_label = y
        month = df_raw.iloc[3, c]
        if pd.notna(month):
            credit_val = df_raw.iloc[55, c]     # Credit is the FIRST of each 3-column block
            if pd.notna(credit_val):
                rows.append({"fiscal_year": str(fy_label).strip(),
                             "month": str(month).strip(),
                             "cumulative_npr_million": float(credit_val)})
        c += 3
    return pd.DataFrame(rows)

bpm6 = extract_bpm6(df6_raw)
bpm6.head()

,fiscal_year,month,cumulative_npr_million
0,2022/23R,Aug,94498.755905
1,2022/23R,Sep,192356.558334
2,2022/23R,Oct,290338.916892
3,2022/23R,Nov,390310.983658
4,2022/23R,Dec,493970.125601


In [17]:
# This should now match exactly (it's the Credit-vs-Credit comparison, not Credit-vs-Net)
check5 = bpm5[bpm5.fiscal_year.str.contains("2022/23")].iloc[0]
check6 = bpm6[bpm6.fiscal_year.str.contains("2022/23")].iloc[0]
print(check5.cumulative_npr_million, check6.cumulative_npr_million)

94498.75590496678 94498.75590496678


In [18]:
MONTH_FIX = {"June": "Jun", "July": "Jul"}
for df in (bpm5, bpm6):
    df["month"] = df["month"].str.strip().replace(MONTH_FIX)

# BPM5 and BPM6 both cover FY2022/23 and FY2023/24 — keep BPM5's version,
# only pull *new* fiscal years from BPM6 to avoid duplicating rows
bpm5_fys = set(bpm5["fiscal_year"].str.replace(r"\s*[RP]$", "", regex=True))
bpm6["fy_clean"] = bpm6["fiscal_year"].str.replace(r"\s*[RP]$", "", regex=True)
bpm6_new = bpm6[~bpm6["fy_clean"].isin(bpm5_fys)].drop(columns="fy_clean")

combined = pd.concat([bpm5, bpm6_new], ignore_index=True)
combined.shape

(304, 3)

In [19]:
FY_MONTH_ORDER = ["Aug","Sep","Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul"]

def fy_month_to_date(fy_label, month_label):
    fy_start_year = int(fy_label.strip()[:4])
    idx = FY_MONTH_ORDER.index(month_label)
    calendar_year = fy_start_year + (0 if idx <= 4 else 1)   # Aug-Dec stay in start year, Jan-Jul roll to next
    month_num = [8,9,10,11,12,1,2,3,4,5,6,7][idx]
    return pd.Timestamp(year=calendar_year, month=month_num, day=1)

combined["date"] = combined.apply(lambda r: fy_month_to_date(r["fiscal_year"], r["month"]), axis=1)
combined = combined.sort_values("date").drop_duplicates(subset="date").reset_index(drop=True)
combined.head()

,fiscal_year,month,cumulative_npr_million,date
0,2000/01,Aug,2445.9,2000-08-01
1,2000/01,Sep,5295.8,2000-09-01
2,2000/01,Oct,11452.4,2000-10-01
3,2000/01,Nov,15370.7,2000-11-01
4,2000/01,Dec,19230.4,2000-12-01


In [20]:
combined["fy_clean"] = combined["fiscal_year"].str.replace(r"\s*[RP]$", "", regex=True)
combined["remittance_npr_million"] = combined.groupby("fy_clean")["cumulative_npr_million"].diff()

first_month = combined.groupby("fy_clean").cumcount() == 0
combined.loc[first_month, "remittance_npr_million"] = combined.loc[first_month, "cumulative_npr_million"]

combined[["date","fiscal_year","cumulative_npr_million","remittance_npr_million"]].head(14)

,date,fiscal_year,cumulative_npr_million,remittance_npr_million
0,2000-08-01,2000/01,2445.9,2445.9
1,2000-09-01,2000/01,5295.8,2849.9
2,2000-10-01,2000/01,11452.4,6156.6
3,2000-11-01,2000/01,15370.7,3918.3
4,2000-12-01,2000/01,19230.4,3859.7
5,2001-01-01,2000/01,23154.2,3923.8
6,2001-02-01,2000/01,27160.2,4006.0
7,2001-03-01,2000/01,31220.8,4060.6
8,2001-04-01,2000/01,35427.7,4206.9
9,2001-05-01,2000/01,39238.9,3811.2


In [21]:
for fy, grp in combined.groupby("fy_clean"):
    if len(grp) == 12:
        derived = grp["remittance_npr_million"].sum()
        reported = grp["cumulative_npr_million"].iloc[-1]
        assert abs(derived - reported) < 1, f"Mismatch in {fy}: {derived} vs {reported}"
print("All fiscal years validated — derived monthly totals match NRB's reported annual figures.")

All fiscal years validated — derived monthly totals match NRB's reported annual figures.


In [23]:
final = combined[["fiscal_year","month","date","cumulative_npr_million","remittance_npr_million"]]
final.to_csv("remittance_monthly.csv", index=False)
print(f"{len(final)} rows, {final['date'].min().date()} to {final['date'].max().date()}")
final.tail()

304 rows, 2000-08-01 to 2025-11-01


,fiscal_year,month,date,cumulative_npr_million,remittance_npr_million
299,2024/25R,Jul,2025-07-01,1.723270e+06,189113.329934
300,2025/26P,Aug,2025-08-01,1.774116e+05,177411.577549
301,2025/26P,Sep,2025-09-01,3.520847e+05,174673.077157
302,2025/26P,Oct,2025-10-01,5.533082e+05,201223.580321
303,2025/26P,Nov,2025-11-01,6.871287e+05,133820.440540
